In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv


In [2]:
load_dotenv()

model = ChatGroq(model='llama-3.1-8b-instant')

In [3]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    rating: str

In [4]:
def create_outline(state: BlogState) -> BlogState:
    #fetch title
    title = state["title"]

    #call llm get outline
    prompt =f"generate a detailed outline for a blog on the topic - {title}"
    outline = model.invoke(prompt).content

    #update state 
    state["outline"] = outline

    return state

In [5]:
def create_blog(state: BlogState) -> BlogState:

    title = state["title"]
    outline = state["outline"]

    prompt = f"write a detailed blog on the title - {title} using the following outline \n {outline}"

    content = model.invoke(prompt).content

    state ["content"] = content

    return state

In [6]:
def evaluate_blog(state: BlogState) -> BlogState:

    #title = ["title"]
    content = state['content']

    prompt = f"""
    Evaluate the following blog and give:
    1. Rating out of 10
    2. Short feedback

    Blog:
    {content}
    """
    rating = model.invoke(prompt).content

    state["rating"] = rating

    return state 

In [7]:
# create graph

graph = StateGraph(BlogState)

#node
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)
graph.add_node("evaluate_blog", evaluate_blog)

#edges 
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", "evaluate_blog")
graph.add_edge("evaluate_blog", END)

#compile

workflow = graph.compile()

In [8]:
initial_state = {"title": "rise of ai in india"}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'rise of ai in india', 'outline': "**Title:** The Rise of AI in India: Opportunities, Challenges, and Future Prospects\n\n**I. Introduction**\n\n* Brief overview of AI's global significance and India's growing importance in the AI landscape\n* Thesis statement: India is poised to become a leading player in the global AI market, driven by government initiatives, technological advancements, and a growing talent pool.\n\n**II. Current State of AI in India**\n\n* Overview of India's AI ecosystem, including key players, startups, and research institutions\n* Discussion of AI adoption in various sectors, such as:\n\t+ Healthcare\n\t+ Finance\n\t+ Education\n\t+ Manufacturing\n\t+ Agriculture\n* Statistics and trends highlighting India's growing AI adoption rates\n\n**III. Government Initiatives and Policies**\n\n* Overview of government initiatives and policies promoting AI development and adoption in India, such as:\n\t+ Digital India program\n\t+ Make in India initiative\n\t+ AI 

In [9]:
print(final_state["outline"])

**Title:** The Rise of AI in India: Opportunities, Challenges, and Future Prospects

**I. Introduction**

* Brief overview of AI's global significance and India's growing importance in the AI landscape
* Thesis statement: India is poised to become a leading player in the global AI market, driven by government initiatives, technological advancements, and a growing talent pool.

**II. Current State of AI in India**

* Overview of India's AI ecosystem, including key players, startups, and research institutions
* Discussion of AI adoption in various sectors, such as:
	+ Healthcare
	+ Finance
	+ Education
	+ Manufacturing
	+ Agriculture
* Statistics and trends highlighting India's growing AI adoption rates

**III. Government Initiatives and Policies**

* Overview of government initiatives and policies promoting AI development and adoption in India, such as:
	+ Digital India program
	+ Make in India initiative
	+ AI for All program
	+ National Policy on Artificial Intelligence
* Discussion o

In [12]:
print(final_state["rating"])

**Rating: 8/10**

**Feedback:**

The blog provides a comprehensive overview of the current state of AI in India, government initiatives, key players and startups, challenges and concerns, and future prospects and opportunities. The blog is well-structured and easy to follow, with clear headings and concise paragraphs.

The strengths of the blog include:

* Comprehensive coverage of the current state of AI in India, including key statistics and trends.
* In-depth analysis of government initiatives and policies promoting AI adoption and development in India.
* Discussion of emerging AI applications and use cases in India, including healthcare, financial services, education, and smart cities.
* Useful resources for further reading and exploration, including government policies and reports.

However, there are some areas for improvement:

* The blog could benefit from more concrete examples and case studies to illustrate the impact of AI in various sectors in India.
* Some of the sections,